In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

In [15]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

TARGET = 'salary_mean_net'
ID_COL = 'id'

y = train[TARGET]

train_features = train.drop(columns=[TARGET])

full = pd.concat(
    [train_features, test],
    axis=0
).reset_index(drop=True)

full = full.replace([
    'не указано',
    'Не указано',
    'не заполнено'
], np.nan)

print(train.shape)
print(test.shape)

(49051, 26)
(12263, 25)


In [16]:
text_cols = [
    'name',
    'employer_name',
    'key_skills_name',
    'professional_roles_name',
    'specializations_profarea_name',
    'raw_description',
    'raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'lemmaized_wo_stopwords_raw_branded_description'
]

for col in text_cols:

    if col not in full.columns:
        full[col] = ''

    full[col] = full[col].fillna('').astype(str)

full['all_text'] = (
    full['name'] + ' ' +
    full['employer_name'] + ' ' +
    full['professional_roles_name'] + ' ' +
    full['specializations_profarea_name'] + ' ' +
    full['lemmaized_wo_stopwords_raw_description'] + ' ' +
    full['lemmaized_wo_stopwords_raw_branded_description']
)

In [17]:
text_source = full['raw_description'].fillna('').astype(str)

full['desc_len'] = text_source.str.len()

full['desc_words'] = text_source.apply(
    lambda x: len(x.split())
)

full['uppercase_ratio'] = text_source.apply(
    lambda x: sum(c.isupper() for c in x) / (len(x) + 1)
)

full['digit_count'] = text_source.apply(
    lambda x: sum(c.isdigit() for c in x)
)

full['exclamation_count'] = text_source.str.count('!')

full['question_count'] = text_source.str.count(r'\?')

full['has_salary_words'] = text_source.str.contains(
    'зарплат|оклад|доход|income|salary',
    case=False,
    regex=True
).astype(int)

full['has_remote'] = text_source.str.contains(
    'удален|remote|гибрид',
    case=False,
    regex=True
).astype(int)

full['is_moscow'] = (
    full['unified_address_city']
    .fillna('')
    .astype(str)
    .str.lower()
    .str.contains('москва')
).astype(int)

full['name_len'] = (
    full['name']
    .fillna('')
    .astype(str)
    .str.len()
)

full['skills_len'] = (
    full['key_skills_name']
    .fillna('')
    .astype(str)
    .str.len()
)

full['skills_count'] = (
    full['key_skills_name']
    .fillna('')
    .astype(str)
    .apply(lambda x: len(x.split(',')))
)

In [18]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(
    full['all_text']
)

svd = TruncatedSVD(
    n_components=128,
    random_state=42
)

svd_features = svd.fit_transform(
    tfidf_matrix
)

svd_features = pd.DataFrame(
    svd_features,
    columns=[f'svd_{i}' for i in range(128)]
)

full = pd.concat(
    [
        full.reset_index(drop=True),
        svd_features.reset_index(drop=True)
    ],
    axis=1
)

In [19]:
high_cardinality = [
    'employer_name',
    'name_clean',
    'employer_industries',
    'professional_roles_name',
    'specializations_profarea_name'
]

kf_te = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for col in high_cardinality:

    if col not in full.columns:
        continue

    full[f'{col}_te'] = np.nan

    for train_idx, valid_idx in kf_te.split(train):

        X_train_fold = train.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]

        stats = pd.DataFrame({
            col: X_train_fold[col].fillna('missing').astype(str),
            'target': y_train_fold
        })

        global_mean = y.mean()

        agg = stats.groupby(col)['target'].agg(
            ['mean', 'count']
        )

        smooth = (
            (agg['mean'] * agg['count'] + global_mean * 30) /
            (agg['count'] + 30)
        )

        mapping = smooth.to_dict()

        valid_values = (
            train.iloc[valid_idx][col]
            .fillna('missing')
            .astype(str)
            .map(mapping)
            .fillna(global_mean)
        )

        full.loc[valid_idx, f'{col}_te'] = valid_values.values

    stats_full = pd.DataFrame({
        col: train[col].fillna('missing').astype(str),
        'target': y
    })

    agg_full = stats_full.groupby(col)['target'].agg(
        ['mean', 'count']
    )

    smooth_full = (
        (agg_full['mean'] * agg_full['count'] + global_mean * 30) /
        (agg_full['count'] + 30)
    )

    mapping_full = smooth_full.to_dict()

    test_idx = np.arange(len(train), len(full))

    full.loc[test_idx, f'{col}_te'] = (
        full.loc[test_idx, col]
        .fillna('missing')
        .astype(str)
        .map(mapping_full)
        .fillna(global_mean)
    )

In [20]:
drop_cols = [
    'all_text'
]

full = full.drop(columns=drop_cols)

obj_cols = full.select_dtypes(include='object').columns

for col in obj_cols:

    full[col] = (
        full[col]
        .fillna('missing')
        .astype(str)
    )

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    full[[col]] = encoder.fit_transform(
        full[[col]]
    )

In [21]:
train_final = full.iloc[:len(train)].copy()
test_final = full.iloc[len(train):].copy()

test_ids = test_final[ID_COL]

train_final = train_final.drop(
    columns=[ID_COL],
    errors='ignore'
)

test_final = test_final.drop(
    columns=[ID_COL],
    errors='ignore'
)

train_final = train_final.replace(
    [np.inf, -np.inf],
    np.nan
)

test_final = test_final.replace(
    [np.inf, -np.inf],
    np.nan
)

imputer = SimpleImputer(strategy='median')

train_final = pd.DataFrame(
    imputer.fit_transform(train_final),
    columns=train_final.columns
)

test_final = pd.DataFrame(
    imputer.transform(test_final),
    columns=test_final.columns
)

print(train_final.shape)
print(test_final.shape)

(49051, 169)
(12263, 169)


In [22]:
q1 = y.quantile(0.01)
q2 = y.quantile(0.99)

mask = (
    (y >= q1) &
    (y <= q2)
)

train_final = train_final[mask]
y = y[mask]

y_log = np.log1p(y)

print(train_final.shape)

(48107, 169)


In [23]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(train_final))
test_preds = np.zeros(len(test_final))

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(train_final)):

    X_train = train_final.iloc[train_idx]
    X_valid = train_final.iloc[valid_idx]

    y_train = y_log.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=3500,
        learning_rate=0.03,
        depth=8,
        loss_function='RMSE',
        eval_metric='MAPE',
        random_seed=42,
        bagging_temperature=0.5,
        random_strength=1,
        l2_leaf_reg=5,
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(
            X_valid,
            np.log1p(y_valid)
        ),
        use_best_model=True
    )

    preds_valid = np.expm1(
        model.predict(X_valid)
    )

    preds_test = np.expm1(
        model.predict(test_final)
    )

    oof[valid_idx] = preds_valid

    test_preds += preds_test / kf.n_splits

    score = mean_absolute_percentage_error(
        y.iloc[valid_idx],
        preds_valid
    )

    scores.append(score)

    print(f'Fold {fold + 1}: {score}')

print('CV:', np.mean(scores))

0:	learn: 0.0383022	test: 0.0384186	best: 0.0384186 (0)	total: 179ms	remaining: 10m 24s
200:	learn: 0.0219887	test: 0.0230245	best: 0.0230245 (200)	total: 32.3s	remaining: 8m 49s
400:	learn: 0.0202143	test: 0.0218422	best: 0.0218422 (400)	total: 59.2s	remaining: 7m 37s
600:	learn: 0.0188950	test: 0.0212131	best: 0.0212131 (600)	total: 1m 28s	remaining: 7m 4s
800:	learn: 0.0179041	test: 0.0208369	best: 0.0208369 (800)	total: 2m 5s	remaining: 7m 2s
1000:	learn: 0.0170690	test: 0.0205739	best: 0.0205739 (1000)	total: 2m 42s	remaining: 6m 45s
1200:	learn: 0.0163204	test: 0.0203725	best: 0.0203725 (1200)	total: 3m 16s	remaining: 6m 15s
1400:	learn: 0.0156676	test: 0.0202145	best: 0.0202145 (1400)	total: 3m 43s	remaining: 5m 34s
1600:	learn: 0.0150551	test: 0.0200889	best: 0.0200889 (1600)	total: 4m 6s	remaining: 4m 52s
1800:	learn: 0.0145137	test: 0.0199916	best: 0.0199916 (1800)	total: 4m 30s	remaining: 4m 15s
2000:	learn: 0.0139751	test: 0.0198932	best: 0.0198932 (2000)	total: 4m 56s	rema

In [25]:
submission = pd.DataFrame({
    'id': test_ids,
    'salary_mean_net': test_preds
})

submission.to_csv(
    'submission.csv',
    index=False
)

display(submission.head())

,id,salary_mean_net
49051,46224201,105430.760513
49052,42119402,49587.661249
49053,45716401,25651.942424
49054,43716203,23428.406190
49055,47109602,48831.941135
